## Full modular zero-shot pipeline with BLEU, chrF++, and COMET. On Kaggle: enable the GPU accelerator and internet (NLLB + COMET both download from the hub).

## 1. Install & Imports


In [1]:
!pip install transformers

In [2]:
import subprocess, sys

def pip(args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args.split(),
                   check=True)

# Pin a known-good COMET + its dependency stack
pip("unbabel-comet==2.2.2")
pip("datasets==2.19.0")
pip("fsspec==2024.3.1")

print("Done — now RESTART THE KERNEL before continuing.")

Done — now RESTART THE KERNEL before continuing.


In [3]:
!pip install sacrebleu

In [4]:
import torch
import pandas as pd
import sacrebleu
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


# 2. Configs

In [5]:
# ============================================================
# CONFIG
# ============================================================
class Config:
    # --- Model ---
    # Options: facebook/nllb-200-distilled-600M
    #          facebook/nllb-200-distilled-1.3B
    #          facebook/nllb-200-3.3B
    MODEL_PATH = "facebook/nllb-200-3.3B"

    # --- COMET model (reference-based, multilingual) ---
    COMET_MODEL = "Unbabel/wmt22-comet-da"

    # --- NLLB language codes (FLORES-200 format) ---
    HI_CODE = "hin_Deva"   # Hindi (Devanagari)
    AR_CODE = "arb_Arab"   # Modern Standard Arabic

    # --- Data: single CSV with "ar" and "hi" columns ---
    DATA_CSV = "/kaggle/input/datasets/mishbhaul/devtest-parallel-corpus/devtest_parallel_corpus.csv"

    # --- Generation ---
    BATCH_SIZE      = 16     # lower to 8 if you hit OOM
    NUM_BEAMS       = 4
    MAX_LENGTH      = 256
    COMET_BATCH     = 16     # COMET runs its own batching

    # --- Output ---
    OUTPUT_DIR = "/kaggle/working"


CFG = Config()

# 3. Data Loading

In [6]:
# ============================================================
# DATA LOADING
# ============================================================
def load_parallel_csv(path, hi_col="hi", ar_col="ar"):
    """Load aligned Hindi/Arabic sentences from a single CSV.
    Returns (hindi_list, arabic_list)."""
    df = pd.read_csv(path, encoding="utf-8")

    missing = {hi_col, ar_col} - set(df.columns)
    assert not missing, (
        f"CSV missing columns: {missing}. Found: {list(df.columns)}"
    )

    n_before = len(df)
    df = df.dropna(subset=[hi_col, ar_col])
    df[hi_col] = df[hi_col].astype(str).str.strip()
    df[ar_col] = df[ar_col].astype(str).str.strip()
    df = df[(df[hi_col] != "") & (df[ar_col] != "")]
    n_after = len(df)

    print(f"Loaded {n_after} valid pairs from {path} "
          f"(dropped {n_before - n_after})")
    return df[hi_col].tolist(), df[ar_col].tolist()

# 4. Model Loading

In [7]:
# ============================================================
# MODEL LOADING
# ============================================================
def load_nllb(model_path):
    """Load NLLB tokenizer + model onto the available device."""
    print(f"Loading {model_path} ...")
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    ).to(DEVICE)
    model.eval()
    print("NLLB loaded.")
    return tokenizer, model


def load_comet(comet_model):
    """Download + load the COMET evaluation model."""
    from comet import download_model, load_from_checkpoint
    print(f"Loading COMET: {comet_model} ...")
    ckpt = download_model(comet_model)
    model = load_from_checkpoint(ckpt)
    print("COMET loaded.")
    return model

# 5. Translation Function

In [8]:
# ============================================================
# TRANSLATION
# ============================================================
@torch.no_grad()
def translate(sentences, tokenizer, model, src_lang, tgt_lang, cfg):
    """Batch-translate from src_lang to tgt_lang (NLLB FLORES codes)."""
    tokenizer.src_lang = src_lang
    bos_id = tokenizer.convert_tokens_to_ids(tgt_lang)

    outputs = []
    for i in range(0, len(sentences), cfg.BATCH_SIZE):
        batch = sentences[i : i + cfg.BATCH_SIZE]
        enc = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=cfg.MAX_LENGTH,
        ).to(DEVICE)

        gen = model.generate(
            **enc,
            forced_bos_token_id=bos_id,
            num_beams=cfg.NUM_BEAMS,
            max_length=cfg.MAX_LENGTH,
        )
        outputs.extend(
            tokenizer.batch_decode(gen, skip_special_tokens=True)
        )
        done = min(i + cfg.BATCH_SIZE, len(sentences))
        print(f"  {done}/{len(sentences)}", end="\r")

    print()
    return outputs

# 6. Evaluation Metrics

In [9]:
# ============================================================
# EVALUATION
# ============================================================
def eval_surface(hypotheses, references):
    """sacreBLEU BLEU + chrF++.
    chrF++ = chrF with word n-grams (word_order=2). It's the
    standard WMT variant and more reliable than BLEU for
    morphologically rich targets like Arabic and Hindi."""
    bleu  = sacrebleu.corpus_bleu(hypotheses, [references])
    chrfpp = sacrebleu.corpus_chrf(hypotheses, [references],
                                   word_order=2)   # the "++"
    return bleu.score, chrfpp.score


def eval_comet(sources, hypotheses, references, comet_model, cfg):
    """Reference-based COMET score (0-1, higher better)."""
    data = [
        {"src": s, "mt": h, "ref": r}
        for s, h, r in zip(sources, hypotheses, references)
    ]
    out = comet_model.predict(
        data,
        batch_size=cfg.COMET_BATCH,
        gpus=1 if DEVICE == "cuda" else 0,
    )
    return out["system_score"], out["scores"]   # system avg, per-sentence


def evaluate(sources, hypotheses, references, direction_name,
             comet_model, cfg):
    """Run all three metrics for one direction."""
    bleu, chrfpp = eval_surface(hypotheses, references)
    comet_sys, comet_each = eval_comet(
        sources, hypotheses, references, comet_model, cfg
    )

    print(f"\n=== {direction_name} ===")
    print(f"BLEU   : {bleu:.2f}")
    print(f"chrF++ : {chrfpp:.2f}")
    print(f"COMET  : {comet_sys:.4f}")

    return {
        "direction": direction_name,
        "bleu": round(bleu, 2),
        "chrf++": round(chrfpp, 2),
        "comet": round(comet_sys, 4),
    }, comet_each


def save_outputs(src, hyp, ref, comet_each, direction_tag, cfg):
    """Per-sentence dump with COMET score for error inspection."""
    df = pd.DataFrame({
        "source": src,
        "hypothesis": hyp,
        "reference": ref,
        "comet": [round(c, 4) for c in comet_each],
    })
    path = f"{cfg.OUTPUT_DIR}/zeroshot_{direction_tag}.csv"
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved outputs -> {path}")
    return df

# 7. Main Function

In [10]:
import pandas as pd
# ============================================================
# MAIN
# ============================================================
def run():
    # Load data once (hi and ar are aligned)
    hindi, arabic = load_parallel_csv(CFG.DATA_CSV)

    # Load models once
    tokenizer, nllb = load_nllb(CFG.MODEL_PATH)
    comet = load_comet(CFG.COMET_MODEL)

    results = []

    # ---- Hindi -> Arabic ----
    hyp_ar = translate(hindi, tokenizer, nllb,
                       CFG.HI_CODE, CFG.AR_CODE, CFG)
    row, comet_each = evaluate(hindi, hyp_ar, arabic,
                               "Hindi -> Arabic", comet, CFG)
    results.append(row)
    save_outputs(hindi, hyp_ar, arabic, comet_each, "hi2ar", CFG)

    # ---- Arabic -> Hindi ----
    hyp_hi = translate(arabic, tokenizer, nllb,
                       CFG.AR_CODE, CFG.HI_CODE, CFG)
    row, comet_each = evaluate(arabic, hyp_hi, hindi,
                               "Arabic -> Hindi", comet, CFG)
    results.append(row)
    save_outputs(arabic, hyp_hi, hindi, comet_each, "ar2hi", CFG)

    # ---- Summary ----
    summary = pd.DataFrame(results)
    summary.to_csv(f"{CFG.OUTPUT_DIR}/zeroshot_summary.csv", index=False)
    print("\n" + "=" * 45)
    print(summary.to_string(index=False))
    return summary


summary = run()

Loaded 500 valid pairs from /kaggle/input/datasets/mishbhaul/devtest-parallel-corpus/devtest_parallel_corpus.csv (dropped 0)
Loading facebook/nllb-200-3.3B ...


tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

COMET loaded.
  500/500


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100


=== Hindi -> Arabic ===
BLEU   : 13.49
chrF++ : 45.07
COMET  : 0.8529
Saved outputs -> /kaggle/working/zeroshot_hi2ar.csv
  500/500


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100


=== Arabic -> Hindi ===
BLEU   : 24.37
chrF++ : 48.30
COMET  : 0.7149
Saved outputs -> /kaggle/working/zeroshot_ar2hi.csv

      direction  bleu  chrf++  comet
Hindi -> Arabic 13.49   45.07 0.8529
Arabic -> Hindi 24.37   48.30 0.7149
